# PySpark Structured Streaming — Silver Layer Transformation

## Objective

This notebook implements the **real-time Silver-layer processing** for the retail streaming pipeline.

Streaming retail events generated by `event_simulator.py` are continuously ingested into the Lakehouse Bronze layer. This notebook uses **PySpark Structured Streaming** to continuously read the incoming Bronze events, clean and validate the data, apply the required transformations, and write the refined results to the Silver layer as Delta tables.

The purpose of this notebook is to demonstrate how the platform processes continuously arriving data rather than processing a fixed batch of records.

---

## Streaming Data Flow

The streaming transformation follows the architecture:

**Event Simulator → Fabric Real-Time Intelligence → Lakehouse Bronze → PySpark Structured Streaming → Silver → Gold → Power BI**

### Input

The primary Bronze streaming table is:

`1_bronze.bronze_streaming_events`

This table contains the raw retail events ingested from the streaming pipeline.

The Bronze layer preserves the incoming event data with minimal transformation.

---

## Bronze Streaming Schema

The current Bronze streaming event schema contains:

- `event_type`
- `event_timestamp`
- `customer_id`
- `product_id`
- `product_description`
- `country`
- `session_id`
- `view_duration_seconds`
- `EventProcessedUtcTime`
- `PartitionId`
- `EventEnqueuedUtcTime`

---

# Processing Steps

## 1. Inspect the Bronze Streaming Data

Before starting the continuous streaming transformation, inspect the Bronze table to understand:

- Available columns
- Data types
- Event structure
- Event types
- Timestamp formats
- Nullable fields
- Event-specific attributes

Initial inspection is performed using a standard PySpark DataFrame.

---

## 2. Create a Streaming DataFrame

Use PySpark Structured Streaming to continuously read new records from the Bronze Delta table.

The streaming input will use:

`spark.readStream`

rather than:

`spark.read`

This allows Spark to continuously process new events as they arrive.

---

## 3. Clean and Standardize the Streaming Data

Apply Silver-layer transformations to the incoming Streaming DataFrame.

These transformations may include:

- Standardizing column names
- Converting columns to appropriate data types
- Converting `event_timestamp` from string to timestamp
- Standardizing timestamp handling
- Trimming or normalizing string values
- Handling invalid or malformed values
- Removing unnecessary fields where appropriate

---

## 4. Validate Streaming Data Quality

Apply data-quality rules to identify or handle invalid events.

Potential checks include:

- Missing event types
- Invalid timestamps
- Missing customer, product, or session identifiers where required
- Negative or invalid `view_duration_seconds`
- Invalid event types
- Malformed records
- Duplicate events

The objective is to ensure that only valid and analytically useful data progresses into the Silver layer.

---

## 5. Handle Event Time and Late-Arriving Data

Use the retail event timestamp as the primary **event-time** column.

Where appropriate, implement:

- Event-time processing
- Watermarking
- Late-arriving event handling

This is important because streaming systems must account for events that may arrive after their actual event time.

---

## 6. Handle Duplicate Events

Implement an appropriate deduplication strategy for streaming events.

The strategy should prevent the same event from being processed multiple times while taking into consideration the requirements of a continuously running streaming pipeline.

---

## 7. Apply Business-Relevant Transformations

Create additional attributes where they are useful for downstream analytics.

Examples may include:

- Event date
- Event hour
- Event minute
- Processing delay
- Standardized event categories
- Other analytical attributes required by the Gold layer

Transformations should remain appropriate for the Silver layer rather than creating business-level aggregations prematurely.

---

## 8. Write the Refined Data to Silver

Write the transformed Streaming DataFrame to the Lakehouse Silver layer as a **Delta table** using:

`writeStream`

The Silver table will contain cleaned, validated, standardized and enriched streaming events.

Conceptually:

**Bronze Streaming Events → Streaming DataFrame → Transformations → Silver Delta Table**

---

## 9. Configure Streaming Checkpointing

Configure a checkpoint location for the streaming query.

Checkpointing allows Spark Structured Streaming to maintain the progress and state of the streaming computation and supports reliable recovery if the streaming job is interrupted.

---

## 10. Configure the Streaming Query

Configure the streaming write with the appropriate:

- Output mode
- Trigger
- Checkpoint location
- Target Silver table

The query should remain active and continuously process newly arriving events.

---

## 11. Validate the Silver Streaming Output

After starting the streaming query, verify that:

- The Silver table is created successfully
- New Bronze events are being processed
- Records are arriving in Silver
- Data types are correct
- Transformations are applied correctly
- Invalid records are handled according to the defined rules
- Duplicate handling works as expected
- Event timestamps are suitable for downstream analysis

---

# Expected Output

The primary output of this notebook is a refined Silver streaming Delta table containing clean and standardized retail events.

The resulting architecture is:

**Bronze**

`1_bronze.bronze_streaming_events`

↓

**PySpark Structured Streaming**

`readStream → transform → writeStream`

↓

**Silver**

`2_silver.silver_streaming_events`

↓

**Gold**

Real-time business metrics and KPIs

↓

**Power BI**

Real-time visualization and analytics

---

# Key Technologies

- Microsoft Fabric
- Fabric Lakehouse
- OneLake
- PySpark
- PySpark Structured Streaming
- Streaming DataFrames
- Delta Lake
- Event-time processing
- Watermarking
- Checkpointing

---

## Important Design Principle

The Bronze layer preserves the incoming streaming data, while the Silver layer is responsible for continuously refining that data.

**Bronze = raw streaming events**

**Silver = clean, validated and standardized streaming events**

**Gold = business-ready real-time analytics and KPIs**

PySpark Structured Streaming is the processing approach used to continuously move the data from Bronze into Silver.


#### 1. Inspect the Bronze Streaming Data

Before creating the continuous streaming pipeline, we first inspect the existing Bronze streaming table.

A standard PySpark DataFrame is used at this stage because we are only examining the data and its schema. We are not starting the streaming query yet.

The `1_bronze_streaming_events` table contains the raw events ingested from the streaming pipeline and serves as the source for the Silver-layer transformation.

The schema inspection allows us to understand:

- Available columns
- Current data types
- Nullable fields
- Timestamp columns
- Event-specific attributes

This information will be used to determine the transformations and data-quality rules required for the Silver layer.

In [ ]:
df_streaming = spark.read.table("1_bronze.bronze_streaming_events")

df_streaming.printSchema()


StatementMeta(, 65bc3ac5-b065-4cc6-9cbb-b564b228fb89, 8, Finished, Available, Finished, False)

root
 |-- event_type: string (nullable = true)
 |-- event_timestamp: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- product_description: string (nullable = true)
 |-- country: string (nullable = true)
 |-- session_id: string (nullable = true)
 |-- view_duration_seconds: long (nullable = true)
 |-- EventProcessedUtcTime: timestamp (nullable = true)
 |-- PartitionId: long (nullable = true)
 |-- EventEnqueuedUtcTime: timestamp (nullable = true)



#### Inspect Sample Records

After examining the schema, we inspect a small sample of the Bronze streaming events to understand the actual values being received.

This helps us identify:

- The format of `event_timestamp`
- Available event types
- Which fields are populated for different event types
- Potential null or unexpected values
- The structure of the incoming retail events

This inspection will inform the design of the Silver transformations.

In [2]:
display(df_streaming.limit(10))

StatementMeta(, 74c8f6c3-b8d9-440e-8574-7969dca23f60, 4, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 3604d861-060b-46c6-b2ad-65b4959b2cf9)

#### 2. Create the Streaming DataFrame

With the Bronze schema and sample records inspected, we can now create the Streaming DataFrame that will serve as the input to the Silver transformation.

Unlike the initial inspection, which used a standard PySpark DataFrame with `spark.read`, the Silver pipeline will use PySpark Structured Streaming with `spark.readStream`.

This allows Spark to continuously process new records appended to the Bronze streaming Delta table as they arrive.

The Streaming DataFrame will serve as the foundation for the subsequent Silver-layer operations, including data-type standardization, validation, cleaning, enrichment, and event-time processing.

In [ ]:
streaming_df = (
    spark.readStream
         .table("1_bronze.bronze_streaming_events")
)

StatementMeta(, 65bc3ac5-b065-4cc6-9cbb-b564b228fb89, 10, Finished, Available, Finished, False)

#### 3. Clean and Standardize the Streaming Data

The Streaming DataFrame now continuously receives new events from the Bronze layer.

Before writing the data to Silver, the incoming events must be cleaned and standardized.

The initial Silver transformations will:

- Convert `event_timestamp` from string to timestamp
- Standardize the event timestamp for event-time processing
- Trim unnecessary whitespace from string fields
- Preserve event-specific NULL values where the attribute does not apply
- Retain the ingestion and processing timestamps provided by the streaming system
- Prepare the data for subsequent validation, deduplication, and event-time processing

These transformations improve the consistency and usability of the streaming data while preserving the original event information.

In [9]:
from pyspark.sql.functions import col, to_timestamp, trim

silver_streaming_df = (
    streaming_df
    .withColumn(
        "event_timestamp",
        to_timestamp(col("event_timestamp"))
    )
    .withColumn("event_type", trim(col("event_type")))
    .withColumn("customer_id", trim(col("customer_id")))
    .withColumn("product_id", trim(col("product_id")))
    .withColumn("product_description", trim(col("product_description")))
    .withColumn("country", trim(col("country")))
    .withColumn("session_id", trim(col("session_id")))
)

StatementMeta(, 65bc3ac5-b065-4cc6-9cbb-b564b228fb89, 11, Finished, Available, Finished, False)

#### 4. Validate Streaming Data Quality

After standardizing the incoming streaming events, the next stage is to apply data-quality rules.

The objective is to ensure that invalid or unusable events do not unnecessarily progress into the Silver layer.

Validation will focus on:

- Valid event types
- Valid event timestamps
- Required identifiers
- Valid `view_duration_seconds` values
- Event-specific validation rules
- Handling malformed or incomplete records

Because this is a streaming pipeline, validation must be applied continuously as new events arrive.

Valid events will continue through the Silver transformation, while invalid records will be handled according to the defined data-quality strategy.

#### 4.1 Define Streaming Data-Quality Rules

Data-quality validation is applied continuously to incoming streaming events before they are written to the Silver layer.

The validation rules are designed around the characteristics of each event type rather than treating every field as universally required.

### General validation rules

- `event_type` must be present and recognized.
- `event_timestamp` must contain a valid timestamp.
- `customer_id` must be present.
- `product_id` must be present.
- `country` must be present.
- `view_duration_seconds`, when provided, must be greater than or equal to zero.

### Event-specific validation

Some fields are only applicable to particular event types.

For example:

- `session_id` is populated when session information is available, particularly for session-related events such as product views and cart actions. It may legitimately be NULL when the source event does not contain session information.

- `view_duration_seconds` is relevant to product-view events and may legitimately be NULL for other event types or when no viewing duration is provided.

These rules are applied continuously as new events arrive.

In [10]:
from pyspark.sql.functions import col

silver_streaming_df = (
    silver_streaming_df
    .filter(col("event_type").isNotNull())
    .filter(col("event_timestamp").isNotNull())
    .filter(col("customer_id").isNotNull())
    .filter(col("product_id").isNotNull())
    .filter(col("country").isNotNull())
    .filter(
        col("view_duration_seconds").isNull()
        | (col("view_duration_seconds") >= 0)
    )
)

StatementMeta(, 65bc3ac5-b065-4cc6-9cbb-b564b228fb89, 12, Finished, Available, Finished, False)

#### 5. Event-Time Processing and Watermarking

Streaming systems can receive events at a different time from when those events actually occurred.

For this reason, the pipeline distinguishes between:

- `event_timestamp` — when the retail event occurred
- `EventEnqueuedUtcTime` — when the event was enqueued by the streaming system
- `EventProcessedUtcTime` — when the event was processed

The `event_timestamp` column will be treated as the primary **event-time** column for analytical processing.

A watermark will be used to allow the streaming pipeline to handle events that arrive later than their event time while preventing Spark from maintaining state indefinitely.

For this project, a **10-minute watermark** will be used as an initial configuration.

This means Spark will continue to accept events that arrive within the defined lateness threshold while eventually removing older state from the streaming query.

In [11]:
silver_streaming_df = (
    silver_streaming_df
    .withWatermark("event_timestamp", "10 minutes")
)

StatementMeta(, 65bc3ac5-b065-4cc6-9cbb-b564b228fb89, 13, Finished, Available, Finished, False)

### 6. Handle Duplicate Streaming Events

Streaming systems can produce duplicate events due to retries, network interruptions, or repeated event delivery.

Duplicate events can lead to inaccurate downstream metrics, such as inflated purchase counts, revenue, or product-view activity.

To perform streaming deduplication, the pipeline needs a way to identify whether two records represent the same event.

The current Bronze schema does not contain a dedicated `event_id`. Therefore, the event identity will initially be derived from a combination of event attributes that together describe the event.

The event timestamp, event type, customer, product, and session information will be considered when defining the event identity.

The watermark applied in the previous step will limit how long Spark needs to maintain the state required for streaming deduplication.

#### 6.1 Create a Derived Event Key

The Bronze streaming source does not provide a dedicated unique `event_id`.

To support streaming deduplication, a derived event key will therefore be created from the available event attributes.

The key will use the combination of:

- `event_type`
- `event_timestamp`
- `customer_id`
- `product_id`
- `session_id`
- `country`

These attributes collectively describe the incoming event in the current Bronze schema.

The derived key will be hashed using SHA-256 to create a compact identifier for the Silver streaming pipeline.

This is an implementation-level deduplication key rather than a guaranteed globally unique event identifier. In a production system, generating a unique event ID at the source would be preferable.

In [12]:
from pyspark.sql.functions import col, coalesce, lit, sha2, concat_ws

silver_streaming_df = (
    silver_streaming_df
    .withColumn(
        "event_key",
        sha2(
            concat_ws(
                "||",
                coalesce(col("event_type"), lit("")),
                coalesce(col("event_timestamp").cast("string"), lit("")),
                coalesce(col("customer_id"), lit("")),
                coalesce(col("product_id"), lit("")),
                coalesce(col("session_id"), lit("")),
                coalesce(col("country"), lit(""))
            ),
            256
        )
    )
)

StatementMeta(, 65bc3ac5-b065-4cc6-9cbb-b564b228fb89, 14, Finished, Available, Finished, False)

#### 6.2 Streaming Deduplication

The derived `event_key` is used to identify duplicate streaming events.

The watermark defined previously limits the amount of historical state Spark must maintain while performing streaming deduplication.

Events with the same `event_key` within the active watermark period will be treated as duplicates.

This prevents duplicate events from unnecessarily progressing into the Silver layer and subsequently inflating downstream metrics.

In [13]:
silver_streaming_df = (
    silver_streaming_df
    .dropDuplicates(["event_key"])
)

StatementMeta(, 65bc3ac5-b065-4cc6-9cbb-b564b228fb89, 15, Finished, Available, Finished, False)

### 7. Add Silver Analytical Attributes

After cleaning and validating the streaming events, additional attributes can be derived to make the refined Silver data more useful for downstream analytics.

The following attributes will be added:

- `event_date` — the calendar date on which the event occurred.
- `event_hour` — the hour in which the event occurred.
- `processing_delay_seconds` — the approximate time between the event occurring and the event being processed.

These attributes provide useful context for downstream Gold aggregations and also demonstrate how streaming event-time information can be enriched during Silver processing.

Business-level aggregations and KPIs will be created later in the Gold layer.

In [14]:
from pyspark.sql.functions import to_date, hour, unix_timestamp

silver_streaming_df = (
    silver_streaming_df
    .withColumn(
        "event_date",
        to_date(col("event_timestamp"))
    )
    .withColumn(
        "event_hour",
        hour(col("event_timestamp"))
    )
    .withColumn(
        "processing_delay_seconds",
        unix_timestamp(col("EventProcessedUtcTime"))
        - unix_timestamp(col("event_timestamp"))
    )
)

StatementMeta(, 65bc3ac5-b065-4cc6-9cbb-b564b228fb89, 16, Finished, Available, Finished, False)

#### 8. Write the Streaming Data to the Silver Layer

The Silver Streaming DataFrame now contains the required cleaning, validation, event-time processing, deduplication, and enrichment transformations.

The transformed Streaming DataFrame will now be continuously written to the Silver layer as a Delta table.

PySpark Structured Streaming uses `writeStream` to continuously process and write new records as they become available.

A checkpoint location is configured so that Spark can track the progress and state of the streaming query and recover reliably if the query is interrupted.

The streaming query will use append mode because new valid events are continuously added to the Silver table.

### Streaming Configuration

- **Source:** `1_bronze.bronze_streaming_events`
- **Processing:** PySpark Structured Streaming
- **Output:** Silver Delta table
- **Output mode:** Append
- **Checkpointing:** Enabled
- **Trigger:** Process newly available data periodically

In [ ]:
query = (
    silver_streaming_df.writeStream
    .format("delta")
    .outputMode("append")
    .option(
        "checkpointLocation",
        "Files/checkpoints/streaming_silver"
    )
    .trigger(processingTime="10 seconds")
    .toTable("2_silver.silver_streaming_events")
)

StatementMeta(, 65bc3ac5-b065-4cc6-9cbb-b564b228fb89, 17, Finished, Available, Finished, False)

### Validate the Streaming Silver Table

The streaming transformation is now running continuously and writing processed events into the Silver layer.

The Silver table will be validated to confirm that:
- streaming records are being written successfully
- the expected schema is present
- event types are being processed
- derived columns created during transformation are populated

This validation confirms that the Bronze-to-Silver streaming pipeline is operating as expected before downstream analytical processing is implemented.

In [ ]:
silver_check_df = spark.read.table("2_silver.silver_streaming_events")

silver_check_df.printSchema()

StatementMeta(, 832269d3-4858-4011-9415-1c5d47d4ad58, 12, Finished, Available, Finished, False)

root
 |-- event_type: string (nullable = true)
 |-- event_timestamp: timestamp (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- product_description: string (nullable = true)
 |-- country: string (nullable = true)
 |-- session_id: string (nullable = true)
 |-- view_duration_seconds: long (nullable = true)
 |-- EventProcessedUtcTime: timestamp (nullable = true)
 |-- PartitionId: long (nullable = true)
 |-- EventEnqueuedUtcTime: timestamp (nullable = true)
 |-- event_key: string (nullable = true)
 |-- event_date: date (nullable = true)
 |-- event_hour: integer (nullable = true)
 |-- processing_delay_seconds: long (nullable = true)



In [11]:
display(silver_check_df.limit(20))

StatementMeta(, 832269d3-4858-4011-9415-1c5d47d4ad58, 13, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, a8447f07-fb26-4d40-a657-edd63d1fd48e)

In [12]:
silver_check_df.groupBy("event_type").count().orderBy("event_type").show()

StatementMeta(, 832269d3-4858-4011-9415-1c5d47d4ad58, 14, Finished, Available, Finished, False)

+----------------+-----+
|      event_type|count|
+----------------+-----+
|     cart_action| 1115|
|  checkout_start|  689|
|inventory_update|  225|
|    product_view| 1833|
|purchase_success|  688|
+----------------+-----+



In [13]:
query.isActive 

StatementMeta(, 832269d3-4858-4011-9415-1c5d47d4ad58, 15, Finished, Available, Finished, False)

True

In [14]:
query.lastProgress 

StatementMeta(, 832269d3-4858-4011-9415-1c5d47d4ad58, 16, Finished, Available, Finished, False)

{'id': 'b85e6b66-90fb-48f6-ba48-d402c84d4fee',
 'runId': '3b086280-faca-40c6-8919-48e0fa9f7539',
 'name': None,
 'timestamp': '2026-09-03T10:06:40.000Z',
 'batchId': 1,
 'numInputRows': 0,
 'inputRowsPerSecond': 0.0,
 'processedRowsPerSecond': 0.0,
 'durationMs': {'latestOffset': 144, 'triggerExecution': 144},
 'stateOperators': [{'operatorName': 'dedupe',
   'numRowsTotal': 4550,
   'numRowsUpdated': 0,
   'allUpdatesTimeMs': 1972,
   'numRowsRemoved': 0,
   'allRemovalsTimeMs': 2,
   'commitTimeMs': 77018,
   'memoryUsedBytes': 848600,
   'numRowsDroppedByWatermark': 0,
   'numShufflePartitions': 200,
   'numStateStoreInstances': 200,
   'customMetrics': {'numDroppedDuplicateRows': 0,
    'rocksdbBytesCopied': 589655,
    'rocksdbCommitCheckpointLatency': 4925,
    'rocksdbCommitCompactLatency': 0,
    'rocksdbCommitFileSyncLatencyMs': 68232,
    'rocksdbCommitFlushLatency': 1770,
    'rocksdbCommitPauseLatency': 0,
    'rocksdbCommitWriteBatchLatency': 0,
    'rocksdbFilesCopied': 2

In [15]:
print("Streaming query active:", query.isActive)
print(query.lastProgress)

StatementMeta(, 832269d3-4858-4011-9415-1c5d47d4ad58, 17, Finished, Available, Finished, False)

Streaming query active: True
{'id': 'b85e6b66-90fb-48f6-ba48-d402c84d4fee', 'runId': '3b086280-faca-40c6-8919-48e0fa9f7539', 'name': None, 'timestamp': '2026-09-03T10:19:30.000Z', 'batchId': 1, 'numInputRows': 0, 'inputRowsPerSecond': 0.0, 'processedRowsPerSecond': 0.0, 'durationMs': {'latestOffset': 49, 'triggerExecution': 49}, 'stateOperators': [{'operatorName': 'dedupe', 'numRowsTotal': 4550, 'numRowsUpdated': 0, 'allUpdatesTimeMs': 1972, 'numRowsRemoved': 0, 'allRemovalsTimeMs': 2, 'commitTimeMs': 77018, 'memoryUsedBytes': 848600, 'numRowsDroppedByWatermark': 0, 'numShufflePartitions': 200, 'numStateStoreInstances': 200, 'customMetrics': {'numDroppedDuplicateRows': 0, 'rocksdbBytesCopied': 589655, 'rocksdbCommitCheckpointLatency': 4925, 'rocksdbCommitCompactLatency': 0, 'rocksdbCommitFileSyncLatencyMs': 68232, 'rocksdbCommitFlushLatency': 1770, 'rocksdbCommitPauseLatency': 0, 'rocksdbCommitWriteBatchLatency': 0, 'rocksdbFilesCopied': 200, 'rocksdbFilesReused': 0, 'rocksdbGetCount':

In [ ]:
silver_check_df = spark.read.table("2_silver.silver_streaming_events")

silver_check_df.groupBy("event_type").count().orderBy("event_type").show()

StatementMeta(, 832269d3-4858-4011-9415-1c5d47d4ad58, 18, Finished, Available, Finished, False)

+----------------+-----+
|      event_type|count|
+----------------+-----+
|     cart_action| 1115|
|  checkout_start|  689|
|inventory_update|  225|
|    product_view| 1833|
|purchase_success|  688|
+----------------+-----+



In [ ]:
bronze_check_df = spark.read.table("1_bronze.bronze_streaming_events")

print("Bronze record count:", bronze_check_df.count())

StatementMeta(, 832269d3-4858-4011-9415-1c5d47d4ad58, 19, Finished, Available, Finished, False)

Bronze record count: 4550


In [18]:
display(
    bronze_check_df
    .orderBy("EventEnqueuedUtcTime", ascending=False)
    .limit(10)
)

StatementMeta(, 832269d3-4858-4011-9415-1c5d47d4ad58, 20, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 2e66eda4-32d9-4260-88e8-2f1d166d83e6)

In [19]:
display(
    bronze_check_df
    .orderBy("EventEnqueuedUtcTime", ascending=False)
    .limit(10)
)

StatementMeta(, 832269d3-4858-4011-9415-1c5d47d4ad58, 21, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, c2898454-b73a-49e8-a8bc-74c4e40853fa)

In [ ]:
bronze_check_df = spark.read.table("1_bronze.bronze_streaming_events")

print("Bronze record count:", bronze_check_df.count())

StatementMeta(, 832269d3-4858-4011-9415-1c5d47d4ad58, 22, Finished, Available, Finished, False)

Bronze record count: 4550


In [21]:
display(
    bronze_check_df
    .orderBy("EventEnqueuedUtcTime", ascending=False)
    .limit(10)
)

StatementMeta(, 832269d3-4858-4011-9415-1c5d47d4ad58, 23, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 417d48b8-bd71-4355-b1ff-6b44ba701387)

In [ ]:
bronze_check_df = spark.read.table("1_bronze.bronze_streaming_events")

total_rows = bronze_check_df.count()
new_rows = total_rows - 4550

print(f"Previous Bronze rows: 4550")
print(f"Current Bronze rows:  {total_rows}")
print(f"New streaming rows:   {new_rows}")

StatementMeta(, 65bc3ac5-b065-4cc6-9cbb-b564b228fb89, 3, Finished, Available, Finished, False)

Previous Bronze rows: 4550
Current Bronze rows:  5315
New streaming rows:   765


In [ ]:
silver_check_df = spark.read.table("2_silver.silver_streaming_events")

total_silver_rows = silver_check_df.count()

print(f"Current Silver rows: {total_silver_rows}")

StatementMeta(, 65bc3ac5-b065-4cc6-9cbb-b564b228fb89, 4, Finished, Available, Finished, False)

Current Silver rows: 4550


In [3]:
silver_check_df.groupBy("event_type").count().orderBy("event_type").show()

StatementMeta(, 65bc3ac5-b065-4cc6-9cbb-b564b228fb89, 5, Finished, Available, Finished, False)

+----------------+-----+
|      event_type|count|
+----------------+-----+
|     cart_action| 1115|
|  checkout_start|  689|
|inventory_update|  225|
|    product_view| 1833|
|purchase_success|  688|
+----------------+-----+



In [16]:
print("Streaming query active:", query.isActive)
print("Streaming query status:", query.status)

StatementMeta(, 65bc3ac5-b065-4cc6-9cbb-b564b228fb89, 18, Finished, Available, Finished, False)

Streaming query active: True
Streaming query status: {'message': 'Processing new data', 'isDataAvailable': True, 'isTriggerActive': True}


In [ ]:
silver_check_df = spark.read.table("2_silver.silver_streaming_events")

print("Current Silver rows:", silver_check_df.count())

StatementMeta(, 65bc3ac5-b065-4cc6-9cbb-b564b228fb89, 19, Finished, Available, Finished, False)

Current Silver rows: 5980


In [ ]:
silver_check_df = spark.read.table("2_silver.silver_streaming_events")

print("Current Silver rows:", silver_check_df.count())

silver_check_df.groupBy("event_type").count().orderBy("event_type").show()

StatementMeta(, 65bc3ac5-b065-4cc6-9cbb-b564b228fb89, 20, Finished, Available, Finished, False)

Current Silver rows: 6100
+----------------+-----+
|      event_type|count|
+----------------+-----+
|     cart_action| 1525|
|  checkout_start|  929|
|inventory_update|  294|
|    product_view| 2423|
|purchase_success|  929|
+----------------+-----+

